# Create Intuition About Entropy and Information using Compression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import os

One intuition for the entropy of a source is that it is the amount of incompressible information produced by the source. The binary file created by this script contains data from a source with adjustable entropy. Compressing this binary file leads to a file size that is (usually) reduced w.r.t. the original, however, never below the entropy.

First we define some functions to write a binary file with random bits.

In [ ]:
compression_points = []
chunk_size = 8 * 2**20 # 1 MiB
num_chunks = 10
num_bits = chunk_size * num_chunks


def h(p):
    """ Binary entropy function """
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def array_to_bytes(arr: np.ndarray):
    """ Convert an array of 0, 1 into bytes """
    num_bytes = math.ceil(len(arr) / 8)
    padded_arr = np.zeros(num_bytes * 8, dtype=int)
    padded_arr[:len(arr)] = arr[:]
    bits_arr = padded_arr.reshape(num_bytes, 8)
    values_arr = bits_arr * 2**np.arange(8)[::-1]
    bytes_list = [int(val) for val in np.sum(values_arr, axis=1)]
    return bytes(bytes_list)

def bit_to_MiB(num_bits):
    return byte_to_MiB(num_bits / 8)

def byte_to_MiB(num_bytes):
    return num_bytes / 2**20

def mk_random_file(p):
    """ Make a file containing random bits where P(bit = 1) = p """
    rng = np.random.default_rng()

    with open('random.bin', 'bw') as binary_file:
        for _ in range(num_chunks):
            data = rng.binomial(1, p, chunk_size)
            binary_file.write(array_to_bytes(data))
        

We denote the probability for a random bit in the produced file to be one by $p = P(\mathrm{bit}=1)$. Then each bit in the file has entropy $h(p)$, where $h(p) = -p \log_2(p) - (1-p) \log_2(1 - p)$ is the binary entropy function. As all bits in the file are independent, the file has the entropy $\mathbb{H}_\mathrm{file}(p) = \mathtt{num\_bits} \cdot h(p)$, where `num_bits` is the number of bits in the file.

In [ ]:
entropy_curve = (np.linspace(0.001, 0.999), bit_to_MiB(num_bits * h(np.linspace(0.001, 0.999))))
plt.plot(*entropy_curve)
plt.xlabel('p')
plt.ylabel('$H_\\mathrm{file}(p)$ in MiB')

Lets test this ...

In [ ]:
# Create random file
p = 0.2
mk_random_file(p)
print(f'Entropy of file: {num_bits} * {h(p):.3f} bit == {bit_to_MiB(num_bits * h(p)):.2f} MB')

# Uncompressed file size
print(f'Uncompressed file size: {byte_to_MiB(os.path.getsize("random.bin")):5.2f} MiB')

# Compress the random file
!gzip -kf random.bin

# Compressed file size
compressed_size = byte_to_MiB(os.path.getsize("random.bin.gz"))
print(f'Compressed file size:   {compressed_size:5.2f} MiB')

# Compare to entropy curve
compression_points.append((p, compressed_size))

plt.plot(*entropy_curve, label='$H_\\mathrm{file}(p)$')
plt.scatter(*zip(*compression_points), color='tab:orange', label='gzip compression')
plt.xlabel('p')
plt.ylabel('file size in MiB')
plt.legend()
plt.grid()